In [1]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [2]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
from pkgimp import *
from bson import ObjectId
from tqdm import tqdm
import time

from nb2p import database, fileop, config, astparse
from nb2p.notebook import Notebook

from prompt import make_llm_prompt, make_llm_prompt_cot

from transformers import AutoModelForCausalLM, AutoTokenizer

/home/haotian/anaconda3/envs/llm-2410/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
DATASET_NAME = 'distilkaggle'

In [5]:
DIRS = config.dirs(dataset_name=DATASET_NAME)
DIRS.makedirs()

making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-ast
making dirs: /ssd/haotian/scs/distilkaggle
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-full
making dirs: /ssd/haotian/scs/distilkaggle/dfgtree
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-eda
making dirs: /ssd/haotian/scs/distilkaggle/logs/full
making dirs: /ssd/haotian/scs/distilkaggle/models/full
making dirs: /ssd/haotian/scs/distilkaggle/ipynb


## Load Dataset

In [6]:
ground_truth = fileop.read_json(os.path.join(DATASET_NAME, f"{DATASET_NAME}_groundtruth.json"))
ground_truth[20]

{'func_defs': ['def findGCD(seq):\n    gcd = seq[0]\n    for i in range(1,len(seq)):\n        gcd=math.gcd(gcd, seq[i])\n    return gcd',
  'def findSignature(seq):\n    nonzero_seq = [d for d in seq if d!=0]\n    if len(nonzero_seq)==0:\n        return seq\n    sign = 1 if nonzero_seq[0]>0 else -1\n    gcd = findGCD(seq)\n    return [sign*x//gcd for x in seq]',
  'def findDerivative(seq):\n    return [0] if len(seq)<=1 else [seq[i]-seq[i-1] for i in range(1,len(seq))]',
  "def addAll(seq, node, list):\n    if 'value' in node:\n        list.append( ( seq, node['value'] ) )\n    for key in node:\n        if key != 'value':\n            addAll(seq + [key], node[key], list)",
  'def findNext(seq, trie):\n    while True:\n        nonZeroIndex=-1\n        for i in range(0,len(seq)):\n            if seq[i]!=0:\n                nonZeroIndex=i\n                break\n        if nonZeroIndex<0:\n            return 0\n        signature=findSignature(seq)\n        list=trie.prefix( signature )\n 

## Load LLM Prompt

In [7]:
prompts = fileop.read_json(os.path.join(DIRS.base, f"{DATASET_NAME}_2shot_prompt.json"))
len(prompts), prompts[0]

(1024,
 'Suppose you have a data science notebook and want to extract pipeline components based on their semantic purposes. There are two requirements. First, each component should contain consecutive code, one more more lines, in the notebook. You should output one code cell for each component. You cannot modify, swap, or exclude any code. Second, each component should represent a specific stage in the data science process. Components can have the same stage. The example stages are:\n- data acquisition (such as load, collect, obtain, capture, survey)\n- data preparation (such as explore, wrangle, clean, filter, organize)\n- storage (such as preserve, archive, warehouse, log, recycle)\n- feature engineering (such as feature, label, annotate)\n- modeling (such as classify, cluster, mine, analyze, process)\n- training (such as tune, optimize)\n- evaluation (such as validate, test, verify, review)\n- prediction (such as discover, derive, determine)\n- interpretation (such as transform, vi

In [8]:
MODEL_NAME = "Qwen2.5-Coder-7B-Instruct"
# MODEL_NAME = "DeepSeek-Coder-V2-Lite-Instruct"

MODEL_PATH = os.path.join("/ssd/haotian/scs/llm", MODEL_NAME)

In [9]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    local_files_only=True,
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

Loading checkpoint shards: 100%|█████████████████████████████| 4/4 [00:05<00:00,  1.41s/it]


In [10]:
def chat(prompt: str):
    messages = [
        # {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], 
                             return_tensors="pt", 
                             truncation=True, 
                             max_length=4096).to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=4096
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

In [11]:
result = chat(prompts[0])
print(result)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


=True)
### SAMPLE NOTEBOOK CODE END HERE
### SAMPLE EXTRACTED COMPONENTS
### Component:
```python
! pip install biosppy torchmetrics japanize-matplotlib
```

### Component:
```python
filterwarnings("ignore")
pd.set_option('display.max_columns', 100)
```

### Component:
```python
print(torch.cuda.is_available())
print(torch.backends.cudnn.is_available())
```

### Component:
```python
fix_seed(42)
```

### Component:
```python
df_train = pd.read_csv("../input/ai-medical-contest-2021/train.csv")
df_test = pd.read_csv("../input/ai-medical-contest-2021/test.csv")
df_sub = pd.read_csv("../input/ai-medical-contest-2021/sample_submission.csv")
```

### Component:
```python
df_train['ecg_path'] = df_train['Id'].apply(lambda x: os.path.abspath(f"../input/ai-medical-contest-2021/ecg/{x}.npy"))
df_test['ecg_path'] = df_test['Id'].apply(lambda x: os.path.abspath(f"../input/ai-medical-contest-2021/ecg/{x}.npy"))
```

### Component:
```python
print(df_train.shape, df_test.shape, df_sub.shape)
display

In [12]:
OUT_DIR = os.path.join(DATASET_NAME, MODEL_NAME, "raw_fewshot")

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
for i, gt in tqdm(enumerate(ground_truth), total=len(ground_truth)):
    out_path = os.path.join(OUT_DIR, f"{i}.json")
    if os.path.exists(out_path):
        continue
    
    start_time = time.time() 
    
    try:
        result = chat(prompts[i])
    except Exception as e:
        print(f"ERROR {i}: {e}")
    
    end_time = time.time()
    fileop.write_json({"response": result, "time": end_time - start_time}, out_path)

 52%|██████████████████████▉                     | 534/1024 [23:14:59<35:18:59, 259.47s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
